In [1]:
# Jupyter: 커널 → 재시작 및 모든 출력 지우기 후 실행
import tensorflow as tf
print("TF:", tf.__version__)   # 2.18.0

import keras
print("Keras:", keras.__version__)   # 3.x

TF: 2.18.0
Keras: 3.13.2


In [2]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   # keras 호환 플래그 (맨 위에)

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Input, Dropout, Conv1DTranspose
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)   # 2.18.0 출력되면 정상

TensorFlow: 2.18.0


In [3]:
dir(pd)

['ArrowDtype',
 'BooleanDtype',
 'Categorical',
 'CategoricalDtype',
 'CategoricalIndex',
 'DataFrame',
 'DateOffset',
 'DatetimeIndex',
 'DatetimeTZDtype',
 'ExcelFile',
 'ExcelWriter',
 'Flags',
 'Float32Dtype',
 'Float64Dtype',
 'Grouper',
 'HDFStore',
 'Index',
 'IndexSlice',
 'Int16Dtype',
 'Int32Dtype',
 'Int64Dtype',
 'Int8Dtype',
 'Interval',
 'IntervalDtype',
 'IntervalIndex',
 'MultiIndex',
 'NA',
 'NaT',
 'NamedAgg',
 'Period',
 'PeriodDtype',
 'PeriodIndex',
 'RangeIndex',
 'Series',
 'SparseDtype',
 'StringDtype',
 'Timedelta',
 'TimedeltaIndex',
 'Timestamp',
 'UInt16Dtype',
 'UInt32Dtype',
 'UInt64Dtype',
 'UInt8Dtype',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__docformat__',
 '__file__',
 '__git_version__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 '_built_with_meson',
 '_config',
 '_is_numpy_dev',
 '_libs',
 '_pandas_datetime_CAPI',
 '_pandas_parser_CAPI',
 '_testing',
 '_typing',
 '_version_meson',
 'annota

In [5]:
vib_normal_ds = pd.read_csv("data/data/vibration_normal.csv")
vib_normal_ds.head()

,Time,0,3.12,6.25,9.38,12.5,15.62,18.75,21.88,25,...,1568.75,1571.88,1575,1578.12,1581.25,1584.38,1587.5,1590.62,1593.75,1596.88
0,2021-08-02 6:47,0.000448,0.000634,0.000895,0.001124,0.000333,0.000583,0.000242,0.000743,0.001193,...,0.000317,0.000457,0.000994,0.000815,0.000344,0.000280,0.000636,0.000639,0.000780,0.000728
1,2021-08-02 7:12,0.000001,0.000176,0.000315,0.000793,0.001073,0.001130,0.001330,0.000867,0.000609,...,0.000352,0.000346,0.000111,0.000173,0.000533,0.000419,0.000422,0.000654,0.000172,0.000238
2,2021-08-02 7:36,0.000594,0.000379,0.001343,0.000454,0.000517,0.000454,0.000693,0.001282,0.001232,...,0.000143,0.000389,0.000983,0.000983,0.000157,0.000881,0.001318,0.000757,0.000638,0.000872
3,2021-08-02 7:59,0.000168,0.000438,0.000732,0.000812,0.000957,0.000835,0.001051,0.000489,0.000181,...,0.000894,0.000802,0.000983,0.000872,0.000740,0.000222,0.000566,0.000917,0.000849,0.000489
4,2021-08-02 8:27,0.000370,0.000512,0.000656,0.000267,0.000236,0.000236,0.000305,0.000629,0.001426,...,0.000731,0.000453,0.000630,0.000712,0.000496,0.000706,0.000513,0.000119,0.000599,0.000746


In [6]:
vib_anomaly_ds = pd.read_csv("data/data/vibration_anomaly.csv")
vib_anomaly_ds.head()

,Time,0,3.12,6.25,9.38,12.5,15.62,18.75,21.88,25,...,1568.75,1571.88,1575,1578.12,1581.25,1584.38,1587.5,1590.62,1593.75,1596.88
0,2022-01-17 16:53,0.017119,0.031108,0.018228,0.045243,0.032953,0.011007,0.020632,0.022722,0.021065,...,0.003080,0.047822,0.073223,0.029722,0.016643,0.005442,0.044247,0.033641,0.034237,0.047196
1,2022-01-26 14:50,0.008463,0.020489,0.040288,0.011786,0.006749,0.027902,0.036479,0.016292,0.009036,...,0.014840,0.025090,0.025218,0.018054,0.023980,0.018671,0.013548,0.043027,0.054982,0.034446
2,2022-02-10 18:16,0.013359,0.049131,0.076610,0.061570,0.055249,0.043330,0.030105,0.028493,0.044670,...,0.052756,0.061957,0.036153,0.027028,0.055728,0.057243,0.039780,0.034525,0.041085,0.023809
3,2022-02-22 9:21,0.016137,0.034178,0.022711,0.036808,0.054250,0.031692,0.011437,0.006979,0.010247,...,0.009296,0.003761,0.002089,0.007499,0.007337,0.004411,0.001132,0.004809,0.004792,0.006091
4,2022-02-23 19:40,0.002293,0.031036,0.061567,0.057566,0.040840,0.025495,0.012431,0.010275,0.011133,...,0.023630,0.017060,0.008875,0.005707,0.006139,0.007393,0.002851,0.006780,0.009328,0.006557


In [7]:
cur_normal_ds = pd.read_csv("data/data/current_normal.csv")
cur_normal_ds.head()

,Time,0,1.91,3.81,5.72,7.63,9.54,11.44,13.35,15.26,...,1933.93,1935.83,1937.74,1939.65,1941.56,1943.46,1945.37,1947.28,1949.19,1951.09
0,2021-09-01 0:04,0.028044,0.033194,0.023135,0.019984,0.012384,0.015154,0.003928,0.002710,0.001608,...,0.000641,0.000434,0.000695,0.000511,0.000566,0.000929,0.000841,0.000147,0.000633,0.000309
1,2021-09-01 0:07,0.017024,0.061488,0.025492,0.019490,0.003985,0.000198,0.001219,0.000958,0.000525,...,0.000394,0.000437,0.000833,0.001000,0.001070,0.000533,0.000425,0.000545,0.000734,0.000580
2,2021-09-01 0:10,0.055605,0.051026,0.033430,0.015864,0.010201,0.004803,0.001797,0.002638,0.002295,...,0.000610,0.000373,0.000331,0.000151,0.000219,0.000310,0.000346,0.000757,0.000883,0.000536
3,2021-09-01 0:17,0.070640,0.090214,0.061381,0.023377,0.018212,0.004499,0.001965,0.002304,0.002807,...,0.000676,0.000503,0.000430,0.000347,0.000463,0.000607,0.000992,0.000579,0.000623,0.000383
4,2021-09-01 0:21,0.089390,0.101223,0.069492,0.029898,0.013571,0.007979,0.002882,0.003186,0.000395,...,0.000552,0.000507,0.000512,0.000557,0.000603,0.000919,0.000860,0.000146,0.000602,0.000272


In [ ]:
cur_anomaly_ds = pd.read_csv("data/data/current_anomaly.csv")
cur_anomaly_ds.head()

In [ ]:
vib_normal_ds.tail()

In [ ]:
vib_normal_ds[15:20]

In [ ]:
# 시각화 공통 속성 지정. rc(runtime configuration)
plt.rc("font", size=20)
plt.rc("axes", labelsize=20)
plt.rc("xtick", labelsize=20)
plt.rc("ytick", labelsize=20)
plt.rc("legend", fontsize=20)
plt.rc("figure", titlesize=30)

In [ ]:
plt.figure(figsize=(20, 5))
plt.xlabel("Frequency (Hz)")
plt.ylabel("Vibration (g)")
plt.plot(vib_normal_ds.iloc[0, 1:11], marker="o", color="red") # makrer: o, *, +  | color : red, blue, green 
plt.show

In [ ]:
plt.figure(figsize=(20, 5))
plt.xlabel("Frequency (Hz)")
plt.ylabel("Current (A)")
plt.plot(cur_normal_ds.iloc[0, 1:11], marker="*", color="blue")
plt.show

In [ ]:
plt.figure(figsize=(20, 5))
plt.xlabel("Frequency (Hz)")
plt.ylabel("Vibration (g)")
plt.plot(vib_normal_ds.iloc[0, 1:11], marker="o", color="blue", label="Normal") # makrer: o, *, +  | color : red, blue, green 
plt.plot(vib_anomaly_ds.iloc[0, 1:11], marker="o", color="red", label="Anomaly") # makrer: o, *, +  | color : red, blue, green 
plt.legend()
plt.show

In [ ]:
vib_normal_ds.info()

In [ ]:
vib_anomaly_ds.info()

In [ ]:
cur_normal_ds.info()

In [ ]:
cur_anomaly_ds.info()

In [ ]:
vib_normal_ds.describe()

In [ ]:
vib_anomaly_ds.describe()

In [ ]:
cur_normal_ds.describe()

In [ ]:
cur_anomaly_ds.describe()

In [ ]:
vib_train_ds = vib_normal_ds.iloc[:1758, 1:].values
vib_test_ds = pd.concat([vib_normal_ds.iloc[1758:, 1:], vib_anomaly_ds.iloc[:, 1:]], ignore_index=True).values

print("진동 학습데이터 수:", len(vib_train_ds))
print("진동 평가데이터 수:", len(vib_test_ds))

In [ ]:
cur_tarin_ds = cur_normal_ds.iloc[:8170, 1:].values
cur_test_ds = pd.concat([cur_normal_ds.iloc[8170:, 1:], cur_anomaly_ds.iloc[:, 1:]], ignore_index=True).values

print("전류 학습데이터 수:", len(cur_tarin_ds))
print("전류 평가데이터 수:", len(cur_test_ds))

In [ ]:
scalar = StandardScaler()
vib_train_ds = scalar.fit_transform(vib_train_ds)
vib_test_ds = scalar.transform(vib_test_ds)
cur_train_ds = scalar.fit_transform(cur_tarin_ds)
cur_test_ds = scalar.transform(cur_test_ds)

In [ ]:
plt.figure(figsize=(20,5))
plt.title("Vibration - Normal")
plt.plot(vib_train_ds[0], color="red")
plt.show()

In [ ]:
plt.figure(figsize=(20,5))
plt.title("Vibration - Anomaly")
plt.plot(vib_train_ds[-1], color="red")
plt.show()

In [ ]:
plt.figure(figsize=(20,5))
plt.title("Current - Normal")
plt.plot(cur_test_ds[0], color="red")
plt.show()

In [ ]:
plt.figure(figsize=(20,5))
plt.title("Current - Anomaly")
plt.plot(cur_test_ds[-1], color="red")
plt.show()

In [ ]:
type(vib_train_ds)

In [ ]:
type(cur_train_ds)

In [ ]:
type(vib_test_ds)

In [ ]:
type(cur_test_ds)

In [ ]:
vib_train_ds.shape

In [ ]:
vib_test_ds.shape

In [ ]:
cur_train_ds.shape

In [ ]:
cur_test_ds.shape

In [ ]:
type(vib_train_ds.shape)

In [ ]:
vib_train_ds.shape[0]

In [ ]:
vib_train_ds.shape[1]

In [ ]:
vib_train_ds = vib_train_ds.reshape(vib_train_ds.shape[0], vib_train_ds.shape[1],1)
vib_test_ds = vib_test_ds.reshape(vib_test_ds.shape[0], vib_test_ds.shape[1],1)
cur_train_ds = cur_train_ds.reshape(cur_train_ds.shape[0], cur_train_ds.shape[1], 1)
cur_test_ds = cur_test_ds.reshape(cur_test_ds.shape[0], cur_test_ds.shape[1], 1)

print("진동 학습데이터 shape:", vib_train_ds.shape)
print("진동 평가데이터 shape:", vib_test_ds.shape)
print("전류 학습데이터 shape:", cur_train_ds.shape)
print("전류 평가데이터 shape:", cur_test_ds.shape)

In [ ]:
def Conv_AE(input_ds):
    model = Sequential()
    model.add(Input(shape=(input_ds.shape[1], input_ds.shape[2])))
    model.add(Conv1D(filters=64, kernel_size=input_ds.shape[1]//64, padding="same", strides=2, activation="relu"))
    model.add(Conv1D(filters=32, kernel_size=input_ds.shape[1]//64, padding="same", activation="relu"))
    model.add(Conv1DTranspose(filters=64, kernel_size=input_ds.shape[1]//64, padding="same", strides=2, activation="relu"))
    model.add(Conv1DTranspose(filters=1, kernel_size=input_ds.shape[1]//64, padding="same"))

    return model

In [ ]:
vib_model=Conv_AE(vib_train_ds)
vib_model.summary()

In [ ]:
cur_model=Conv_AE(cur_train_ds)
cur_model.summary()

In [ ]:
vib_model.compile(loss="mae", optimizer="adam")
vib_history = vib_model.fit(x=vib_train_ds, y=vib_train_ds, epochs=100, validation_split=0.2, callbacks=[EarlyStopping(monitor="val_loss", patience=10)])

In [ ]:
cur_model.compile(loss="mae", optimizer="adam")
cur_history = cur_model.fit(x=vib_train_ds, y=vib_train_ds, epochs=100, validation_split=0.2, callbacks=[EarlyStopping(monitor="val_loss", patience=10)])

In [ ]:
vib_history.history["loss"]

In [ ]:
vib_history.history["val_loss"]

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(vib_history.history["loss"], color="blue", label="Train", linewidth=2)
plt.plot(vib_history.history["val_loss"], color="red", label="Validation", linewidth=2)
plt.ylabel("Vibration - Loss(MAE)")
plt.xlabel("Epoch")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(cur_history.history["loss"], color="blue", label="Train", linewidth=2)
plt.plot(cur_history.history["val_loss"], color="red", label="Validation", linewidth=2)
plt.ylabel("Current - Loss(MAE)")
plt.xlabel("Epoch")
plt.legend()
plt.show()

In [ ]:
vib_yhat = vib_model.predict(vib_train_ds)
vib_yhat.shape

In [ ]:
cur_yhat = cur_model.predict(cur_train_ds)
cur_yhat.shape

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(vib_train_ds[0], color="blue", label="Target")
plt.plot(vib_yhat[0], color="red", label="복원")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
plt.plot(cur_train_ds[0], color="blue", label="Target")
plt.plot(cur_yhat[0], color="red", label="복원")
plt.legend()
plt.show()

In [ ]:
vib_mae = np.mean(np.abs(vib_yhat - vib_train_ds), axis=1)
vib_mae

In [ ]:
cur_mae = np.mean(np.abs(cur_yhat - cur_train_ds), axis=1)
cur_mae

In [ ]:
plt.figure(figsize=(20,10))
plt.hist(x=vib_mae, bins=200)
plt.xlabel("Vibration Train dataset - MAE")
plt.ylabel("Number of samples")
plt.show()

In [ ]:
plt.figure(figsize=(20,10))
plt.hist(x=cur_mae, bins=200)
plt.xlabel("Current Train dataset - MAE")
plt.ylabel("Number of samples")
plt.show()

In [ ]:
vib_threshold = np.mean(vib_mae) + 3*np.std(vib_mae)
cur_threshold = np.mean(cur_mae) + 3*np.std(cur_mae)

print("진동 데이터 임계치:", vib_threshold)
print("전류 데이터 임계치:", cur_threshold)

In [ ]:
plt.figure(figsize=(20,10))
plt.text(x=0, y=vib_threshold, s="Vibration-Thredhod", fontsize=20, color="red")
plt.hlines(y=vib_threshold, xmin=0, xmax=len(vib_mae), color="red", linestyle="-")
plt.xlabel("Vibration - Train Dataset (1,758)")
plt.ylabel("Vibration - Train MAE")
plt.plot(vib_mae, color="black")
plt.show()

In [ ]:
plt.figure(figsize=(20,10))
plt.text(x=0, y=cur_threshold, s="Current-Thredhod", fontsize=20, color="red")
plt.hlines(y=cur_threshold, xmin=0, xmax=len(cur_mae), color="red", linestyle="-")
plt.xlabel("Current - Train Dataset (8,170)")
plt.ylabel("Current - Train MAE")
plt.plot(vib_mae, color="black")
plt.show()

In [ ]:
vib_yhat_test = vib_model.predict(vib_test_ds)
vib_mae_test = np.mean(np.abs(vib_yhat_test - vib_test_ds), axis=1)
vib_mae_test.shape

In [ ]:
cur_yhat_test = cur_model.predict(cur_test_ds)
cur_mae_test = np.mean(np.abs(cur_yhat_test - cur_test_ds), axis=1)
cur_mae_test.shape

In [ ]:
vib_mae_test = vib_mae_test.flatten()

plt.figure(figsize=(20,10))
plt.text(x=0, y=float(max(vib_mae_test)), s="Normal data", horizontalalignment="left", color="blue")
plt.text(x=32, y=float(max(vib_mae_test)), s="Anomaly data", horizontalalignment="right", color="blue")
plt.vlines(x=15.5, ymin=0, ymax=float(max(vib_mae_test)), color="blue", linestyle="-")
plt.text(x=0, y=float(vib_threshold), s="Vibration-Threshod", fontsize=20, color="red")
plt.hlines(y=float(vib_threshold), xmin=0, xmax=len(vib_mae_test), color="red", linestyle="-")
plt.xlabel("Vibration-Test dataset(32)")
plt.ylabel("Vibration-Test MAE")
plt.plot(vib_mae_test, color="black", marker="*")
plt.show()

In [ ]:
cur_mae_test = cur_mae_test.flatten()

plt.figure(figsize=(20,10))
plt.text(x=0, y=max(cur_mae_test), s="Normal data", horizontalalignment="left", color="blue")
plt.text(x=32, y=max(cur_mae_test), s="Anomaly data", horizontalalignment="right", color="blue")
plt.vlines(x=15.5, ymin=0, ymax=max(cur_mae_test), color="blue", linestyle="-")
plt.text(x=0, y=cur_threshold, s="Current-Threshod", fontsize=20, color="red")
plt.hlines(y=cur_threshold, xmin=0, xmax=len(cur_mae_test), color="red", linestyle="-")
plt.xlabel("Current-Test dataset(32)")
plt.ylabel("Current-Test MAE")
plt.plot(vib_mae_test, color="black", marker="*")
plt.show()